# 03 - Silver: SECOP (Contratos Públicos)

Trabajo Práctico 2 - Transformación Bronze → Silver con reglas de calidad (DQL)

## Objetivo

Tipar y validar `Datos_Empresas.bronze.DE_Semiestructurado_SECOP_Contratos_Api` (115.773 filas, 87 columnas). Las filas que pasan todas las reglas de calidad van a `Datos_Empresas.silver.contratos_secop`; las que incumplen al menos una van a `Datos_Empresas.dql.rechazados_secop` con el motivo del rechazo.

## Transformaciones de tipado

* `fecha_de_firma`, `fecha_de_inicio_del_contrato`, `fecha_de_fin_del_contrato`, `ultima_actualizacion`: texto ISO (`yyyy-MM-ddTHH:mm:ss.SSS`) → `TIMESTAMP`.
* `valor_del_contrato`, `valor_de_pago_adelantado`, `valor_facturado`, `valor_pendiente_de_pago`, `valor_pagado`: texto → `DOUBLE`.
* `dias_adicionados`: texto → `INT`.
* El resto de las ~75 columnas descriptivas (entidad, proveedor, modalidad, representante legal, etc.) se conserva como texto — no aportan valor al tiparlas, son categorías o identificadores.

## Reglas de calidad (rechazo)

| # | Regla | Justificación |
|---|---|---|
| 1 | `id_contrato` vacío | Es la clave del contrato; sin ella no se puede rastrear ni enlazar con Gold ni con RUES. |
| 2 | `id_contrato` duplicado | Regla de unicidad: cada contrato debe tener un único registro. |
| 3 | `valor_del_contrato` no numérico, nulo o `<= 0` | Un contrato no puede tener valor cero o negativo — regla de negocio. |
| 4 | `fecha_de_inicio_del_contrato` posterior a `fecha_de_fin_del_contrato` | Fecha inconsistente: un contrato no puede terminar antes de empezar. |

In [ ]:
%python
from pyspark.sql import functions as F

TABLA_BRONZE = "Datos_Empresas.bronze.DE_Semiestructurado_SECOP_Contratos_Api"
TABLA_SILVER = "Datos_Empresas.silver.contratos_secop"
TABLA_DQL = "Datos_Empresas.dql.rechazados_secop"

df_bronze = spark.table(TABLA_BRONZE)
print(f"Bronze: {df_bronze.count():,} filas, {len(df_bronze.columns)} columnas")

---

## Paso 1: Tipado seguro de columnas

In [ ]:
%python
FORMATO_FECHA_ISO = "yyyy-MM-dd'T'HH:mm:ss.SSS"

# try_to_timestamp (no to_timestamp): con ANSI SQL activado, to_timestamp lanza
# excepción ante una fecha mal formada en vez de devolver NULL.
df_tipado = (
    df_bronze
    .withColumn("id_contrato", F.trim(F.col("id_contrato")))
    .withColumn("documento_proveedor", F.trim(F.col("documento_proveedor")))
    .withColumn("fecha_de_firma", F.expr(f"try_to_timestamp(fecha_de_firma, '{FORMATO_FECHA_ISO}')"))
    .withColumn("fecha_de_inicio_del_contrato", F.expr(f"try_to_timestamp(fecha_de_inicio_del_contrato, '{FORMATO_FECHA_ISO}')"))
    .withColumn("fecha_de_fin_del_contrato", F.expr(f"try_to_timestamp(fecha_de_fin_del_contrato, '{FORMATO_FECHA_ISO}')"))
    .withColumn("ultima_actualizacion", F.expr(f"try_to_timestamp(ultima_actualizacion, '{FORMATO_FECHA_ISO}')"))
    .withColumn("valor_del_contrato", F.col("valor_del_contrato").cast("double"))
    .withColumn("valor_de_pago_adelantado", F.col("valor_de_pago_adelantado").cast("double"))
    .withColumn("valor_facturado", F.col("valor_facturado").cast("double"))
    .withColumn("valor_pendiente_de_pago", F.col("valor_pendiente_de_pago").cast("double"))
    .withColumn("valor_pagado", F.col("valor_pagado").cast("double"))
    .withColumn("dias_adicionados", F.col("dias_adicionados").cast("int"))
)

df_tipado.select("id_contrato", "fecha_de_firma", "fecha_de_inicio_del_contrato", "fecha_de_fin_del_contrato", "valor_del_contrato").show(5, truncate=False)

---

## Paso 2: Aplicar las reglas de calidad y calcular `_motivo_rechazo`

In [ ]:
%python
from pyspark.sql.window import Window

w_contrato = Window.partitionBy("id_contrato")

df_reglas = (
    df_tipado
    .withColumn("_dup_count", F.count("*").over(w_contrato))
    # Regla 1: id_contrato vacío
    .withColumn(
        "_r1",
        F.when((F.col("id_contrato").isNull()) | (F.col("id_contrato") == ""), F.lit("id_contrato_vacio")),
    )
    # Regla 2: id_contrato duplicado (solo aplica si id_contrato no está vacío, para no solaparse con la Regla 1)
    .withColumn(
        "_r2",
        F.when(
            (F.col("id_contrato").isNotNull()) & (F.col("id_contrato") != "") & (F.col("_dup_count") > 1),
            F.lit("id_contrato_duplicado"),
        ),
    )
    # Regla 3: valor_del_contrato inválido
    .withColumn(
        "_r3",
        F.when(
            (F.col("valor_del_contrato").isNull()) | (F.col("valor_del_contrato") <= 0),
            F.lit("valor_contrato_invalido"),
        ),
    )
    # Regla 4: fecha_inicio posterior a fecha_fin
    .withColumn(
        "_r4",
        F.when(
            F.col("fecha_de_inicio_del_contrato") > F.col("fecha_de_fin_del_contrato"),
            F.lit("fecha_inicio_posterior_a_fecha_fin"),
        ),
    )
    .withColumn("_motivo_rechazo", F.concat_ws("; ", F.col("_r1"), F.col("_r2"), F.col("_r3"), F.col("_r4")))
    .withColumn(
        "_motivo_rechazo",
        F.when(F.col("_motivo_rechazo") == "", F.lit(None)).otherwise(F.col("_motivo_rechazo")),
    )
)

---

## Paso 3: Separar aceptados (Silver) de rechazados (DQL)

In [ ]:
%python
columnas_auxiliares = ["_dup_count", "_r1", "_r2", "_r3", "_r4"]

df_silver = (
    df_reglas.filter(F.col("_motivo_rechazo").isNull())
    .drop(*columnas_auxiliares, "_motivo_rechazo", "_ingested_at", "_source")
    .withColumn("_processed_at", F.current_timestamp())
)

df_dql = (
    df_reglas.filter(F.col("_motivo_rechazo").isNotNull())
    .drop(*columnas_auxiliares)
    .withColumn("_fecha_rechazo", F.current_timestamp())
)

total = df_bronze.count()
aceptados = df_silver.count()
rechazados = df_dql.count()
print(f"Bronze: {total:,} | Silver (aceptados): {aceptados:,} | DQL (rechazados): {rechazados:,}")
assert total == aceptados + rechazados, "Se perdieron o duplicaron filas en la separación Silver/DQL"

---

## Paso 4: Persistir (idempotente: `overwrite` recalcula la tabla completa en cada corrida)

In [ ]:
%python
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_SILVER)
df_dql.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_DQL)

print(f"Escrito: {TABLA_SILVER}")
print(f"Escrito: {TABLA_DQL}")

---

## Paso 5: Validación

In [ ]:
%sql
SELECT COUNT(*) AS total_silver FROM Datos_Empresas.silver.contratos_secop;

SELECT COUNT(*) AS total_dql FROM Datos_Empresas.dql.rechazados_secop;

In [ ]:
%sql
SELECT _motivo_rechazo, COUNT(*) AS cantidad
FROM Datos_Empresas.dql.rechazados_secop
GROUP BY _motivo_rechazo
ORDER BY cantidad DESC
LIMIT 20;

In [ ]:
%sql
SELECT id_contrato, nombre_entidad, proveedor_adjudicado, valor_del_contrato, fecha_de_firma, estado_contrato
FROM Datos_Empresas.silver.contratos_secop
LIMIT 10;

---

## Siguiente paso

Continuar con `04_Silver_CIIU.ipynb` y `05_Silver_TRM.ipynb`, y luego `06_Gold_Empresas_Contratos.ipynb`.